In [ ]:
!pip install pandas pyarrow duckdb kaggle "fsspec<2024.2" --quiet

import pandas, pyarrow, duckdb
print(f"✓ pandas  {pandas.__version__}")
print(f"✓ pyarrow {pyarrow.__version__}")
print(f"✓ duckdb  {duckdb.__version__}")

from pathlib import Path
import pandas as pd
import os
import time
import shutil
import statistics
import warnings
import duckdb

warnings.filterwarnings('ignore', message='coroutine.*was never awaited')
warnings.filterwarnings('ignore', message='Task was destroyed but it is pending')

pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', 20)

print("✓ Ambiente pronto")

CURRENT_DIR = Path.cwd()
if CURRENT_DIR.name == "notebooks" and CURRENT_DIR.parent.name == "src":
    PROJECT_DIR = CURRENT_DIR.parent.parent
elif CURRENT_DIR.name == "notebooks":
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR
SAMPLE_DATA_DIR = PROJECT_DIR / "src" / "sample_data"
OUTPUT_DIR = CURRENT_DIR


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
✓ pandas  2.2.2
✓ pyarrow 24.0.0
✓ duckdb  1.5.3
✓ Ambiente pronto


O username aparece no URL do seu perfil: kaggle.com/SEU_USERNAME.

No Google Colab: faça o upload do kaggle.json pelo painel lateral (ícone de pasta → upload) antes de rodar esta célula.

No Jupyter local: coloque o kaggle.json na pasta do projeto e a célula cuida do resto.

Segurança: após terminar o lab, acesse Kaggle → Account → API → Expire Token e gere um novo token para invalidar o anterior.

In [9]:
KAGGLE_JSON = SAMPLE_DATA_DIR / "kaggle.json"
KAGGLE_DIR = Path.home() / ".kaggle"
KAGGLE_TARGET = KAGGLE_DIR / "kaggle.json"

if not KAGGLE_JSON.exists():
    raise FileNotFoundError(
        "kaggle.json nao encontrado em src/sample_data.\n"
        "1. Acesse kaggle.com -> Account -> API -> Create New Token\n"
        "2. Coloque o arquivo kaggle.json em src/sample_data/"
    )

KAGGLE_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy(KAGGLE_JSON, KAGGLE_TARGET)
KAGGLE_TARGET.chmod(0o600)

print("✓ Credenciais Kaggle configuradas")
print("  Origem:", KAGGLE_JSON)
print("  Destino:", KAGGLE_TARGET)

✓ Credenciais Kaggle configuradas
  Origem: /app/src/sample_data/kaggle.json
  Destino: /root/.kaggle/kaggle.json


0.4 Baixar e extrair o dataset Olist

In [ ]:
DATASET_SLUG = 'olistbr/brazilian-ecommerce'
PASTA_DADOS = PROJECT_DIR / 'src' / 'olist_data'

if not os.path.exists(PASTA_DADOS):
    print("Baixando dataset Olist do Kaggle (~41 MB)...")
    os.makedirs(PASTA_DADOS, exist_ok=True)
    !kaggle datasets download -d {DATASET_SLUG} -p {PASTA_DADOS} --unzip --quiet
    print("✓ Download e extração concluídos")
else:
    print("✓ Pasta src/olist_data/ ja existe - pulando download")

print("\nArquivos disponíveis:")
for f in sorted(os.listdir(PASTA_DADOS)):
    sz = os.path.getsize(os.path.join(PASTA_DADOS, f)) / 1024**2
    print(f"  {f:<55} {sz:>7.2f} MB")

Baixando dataset Olist do Kaggle (~41 MB)...
Dataset URL: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
License(s): CC-BY-NC-SA-4.0
✓ Download e extração concluídos

Arquivos disponíveis:
  olist_customers_dataset.csv                                8.62 MB
  olist_geolocation_dataset.csv                             58.44 MB
  olist_order_items_dataset.csv                             14.72 MB
  olist_order_payments_dataset.csv                           5.51 MB
  olist_order_reviews_dataset.csv                           13.78 MB
  olist_orders_dataset.csv                                  16.84 MB
  olist_products_dataset.csv                                 2.27 MB
  olist_sellers_dataset.csv                                  0.17 MB
  product_category_name_translation.csv                      0.00 MB


O Olist é um marketplace brasileiro real. Os dados cobrem pedidos de 2016 a 2018 — reais e anonimizados. São 9 arquivos separados porque é a modelagem clássica de banco transacional (OLTP): cada tabela tem uma responsabilidade e evita redundância.

0.5 Carregar os dados

In [ ]:
print("Carregando arquivos Olist...")

t = time.time()

df_orders = pd.read_csv(PASTA_DADOS / 'olist_orders_dataset.csv')
df_items = pd.read_csv(PASTA_DADOS / 'olist_order_items_dataset.csv')
df_products = pd.read_csv(PASTA_DADOS / 'olist_products_dataset.csv')
df_transl = pd.read_csv(PASTA_DADOS / 'product_category_name_translation.csv')
df_customers = pd.read_csv(PASTA_DADOS / 'olist_customers_dataset.csv')

print(f"✓ CSVs carregados em {time.time()-t:.2f}s")
print(f"  orders:    {df_orders.shape}")
print(f"  items:     {df_items.shape}")
print(f"  products:  {df_products.shape}")
print(f"  customers: {df_customers.shape}")

Carregando arquivos Olist...
✓ CSVs carregados em 0.33s
  orders:    (99441, 8)
  items:     (112650, 7)
  products:  (32951, 9)
  customers: (99441, 5)


0.6 Montar e limpar o DataFrame do lab

In [4]:
print("Montando dataset do lab...")

t = time.time()

# Traduzir categorias para inglês
df_products = df_products.merge(df_transl, on='product_category_name', how='left')
df_products['category'] = df_products['product_category_name_english'].fillna(
                           df_products['product_category_name'])

# Merge principal: 1 linha por item de pedido
df = (df_orders
      .merge(df_items,    on='order_id',    how='inner')
      .merge(df_products[['product_id','category']], on='product_id', how='left')
      .merge(df_customers[['customer_id','customer_state','customer_city']],
             on='customer_id', how='left'))

# Selecionar e renomear colunas úteis para o lab
df = df[[
    'order_id',
    'customer_id',
    'order_status',
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'product_id',
    'category',
    'seller_id',
    'price',
    'freight_value',
    'customer_state',
    'customer_city',
]].rename(columns={'category': 'product_category_name'})

# Limpeza básica
df['order_purchase_timestamp']      = pd.to_datetime(df['order_purchase_timestamp'])
df['order_delivered_customer_date'] = pd.to_datetime(df['order_delivered_customer_date'],
                                                      errors='coerce')
df = df.dropna(subset=['order_purchase_timestamp', 'price'])
df = df[df['price'] > 0]

print(f"✓ Dataset montado em {time.time()-t:.2f}s")
print(f"  Shape final: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
print(f"  Período:     {df['order_purchase_timestamp'].min().date()} "
      f"→ {df['order_purchase_timestamp'].max().date()}")
print(f"  Categorias:  {df['product_category_name'].nunique()} únicas")
print(f"  Estados:     {df['customer_state'].nunique()} únicos")
print(f"  Pedidos:     {df['order_id'].nunique():,} distintos")
print()
print(df[['order_id','product_category_name','price',
          'customer_state','order_purchase_timestamp']].head(5).to_string(index=False))

Montando dataset do lab...
✓ Dataset montado em 0.21s
  Shape final: 112,650 linhas × 12 colunas
  Período:     2016-09-04 → 2018-09-03
  Categorias:  73 únicas
  Estados:     27 únicos
  Pedidos:     98,666 distintos

                        order_id product_category_name  price customer_state order_purchase_timestamp
e481f51cbdc54678b7cc49136f2d6af7            housewares  29.99             SP      2017-10-02 10:56:33
53cdb2fc8bc7dce0b6741e2150273451             perfumery 118.70             BA      2018-07-24 20:41:37
47770eb9100c2d0c44946d9cf07ec65d                  auto 159.90             GO      2018-08-08 08:38:49
949d5b44dbf5de918fe9c16f97b45f8a              pet_shop  45.00             RN      2017-11-18 19:28:06
ad21c59c0840e6cb83a9ceb5573f8159            stationery  19.90             SP      2018-02-13 21:18:39


0.7 Gravar o arquivo Parquet monolítico (baseline)
Este arquivo representa "antes do particionamento" — um único Parquet com todas as linhas. É o baseline contra o qual vamos comparar durante o lab.

In [5]:
print("Gravando Parquet monolítico (baseline)...", end=' ')
t = time.time()
df.to_parquet('pedidos_snappy.parquet', compression='snappy', index=False)
print(f"✓ {time.time()-t:.2f}s")

size_mono_mb = os.path.getsize('pedidos_snappy.parquet') / 1024**2
print(f"Tamanho: {size_mono_mb:.2f} MB  ({df.shape[0]:,} linhas, todas no mesmo arquivo)")
print()
print("=== PRONTO PARA COMEÇAR ===")
print(f"  DataFrame df:            {df.shape[0]:,} linhas × {df.shape[1]} colunas (em memória)")
print(f"  pedidos_snappy.parquet:  {size_mono_mb:.2f} MB (arquivo monolítico no disco)")

Gravando Parquet monolítico (baseline)... ✓ 0.14s
Tamanho: 10.77 MB  (112,650 linhas, todas no mesmo arquivo)

=== PRONTO PARA COMEÇAR ===
  DataFrame df:            112,650 linhas × 12 colunas (em memória)
  pedidos_snappy.parquet:  10.77 MB (arquivo monolítico no disco)


Parte 1 — Preparar a coluna de data
Antes de particionar, precisamos extrair ano e mes da coluna de timestamp. Não é possível particionar por order_purchase_timestamp diretamente — a cardinalidade seria enorme (quase uma partição por pedido). Extraímos a granularidade adequada e usamos essas colunas inteiras como chaves de particionamento.

Célula 1 — Converter timestamp e extrair partições

In [6]:
print("=== PREPARANDO COLUNAS DE PARTIÇÃO ===\n")

df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])

df['ano'] = df['order_purchase_timestamp'].dt.year
df['mes'] = df['order_purchase_timestamp'].dt.month
df['dia'] = df['order_purchase_timestamp'].dt.day

print("Colunas adicionadas: ano, mes, dia")
print()

# Distribuição de dados por ano e mês
dist = df.groupby(['ano','mes']).size().reset_index(name='linhas')
dist['pct'] = (dist['linhas'] / len(df) * 100).round(1)

print(f"{'ANO':<6} {'MÊS':<6} {'LINHAS':>10}  {'PCT':>7}  {'BARRA'}")
print("-"*55)
for _, row in dist.iterrows():
    barra = '█' * int(row['pct'] / 1)
    print(f"{int(row['ano']):<6} {int(row['mes']):<6} {int(row['linhas']):>10,}  {row['pct']:>6.1f}%  {barra}")

print()
print(f"Total: {len(df):,} linhas em {dist['ano'].nunique()} anos")

# Re-gravar incluindo as novas colunas
df.to_parquet('pedidos_snappy.parquet', compression='snappy', index=False)
print("✓ pedidos_snappy.parquet regravado com colunas ano/mes/dia incluídas")

=== PREPARANDO COLUNAS DE PARTIÇÃO ===

Colunas adicionadas: ano, mes, dia

ANO    MÊS        LINHAS      PCT  BARRA
-------------------------------------------------------
2016   9               6     0.0%  
2016   10            363     0.3%  
2016   12              1     0.0%  
2017   1             955     0.8%  
2017   2           1,951     1.7%  █
2017   3           3,000     2.7%  ██
2017   4           2,684     2.4%  ██
2017   5           4,136     3.7%  ███
2017   6           3,583     3.2%  ███
2017   7           4,519     4.0%  ████
2017   8           4,910     4.4%  ████
2017   9           4,831     4.3%  ████
2017   10          5,322     4.7%  ████
2017   11          8,665     7.7%  ███████
2017   12          6,308     5.6%  █████
2018   1           8,208     7.3%  ███████
2018   2           7,672     6.8%  ██████
2018   3           8,217     7.3%  ███████
2018   4           7,975     7.1%  ███████
2018   5           7,925     7.0%  ███████
2018   6           7,078     6.3% 

A distribuição visual mostra que 2016 tem muito poucos dados (o Olist estava iniciando) e que o volume cresce ao longo de 2017 e 2018. Distribuição desigual entre partições significa que algumas terão megabytes e outras kilobytes — isso afeta a eficiência do pruning.

Célula 2 — Por que escolhemos ano + mês como partição?

In [7]:
print("=== POR QUE PARTICIONAR POR ANO + MÊS? ===\n")

colunas_candidatas = {
    'ano':                      df['ano'].nunique(),
    'mes':                      df['mes'].nunique(),
    'ano+mes (combinado)':      df.groupby(['ano','mes']).ngroups,
    'dia':                      df['dia'].nunique(),
    'product_category_name':    df['product_category_name'].nunique(),
    'customer_state':           df['customer_state'].nunique(),
    'order_id (NÃO USAR)':      df['order_id'].nunique(),
    'customer_id (NÃO USAR)':   df['customer_id'].nunique(),
}

print(f"{'COLUNA CANDIDATA':<35} {'VALORES ÚNICOS':>18}  {'AVALIAÇÃO'}")
print("-"*80)

for col, n_unique in colunas_candidatas.items():
    if n_unique <= 500:
        avaliacao = "✓ Boa candidata"
    elif n_unique <= 5000:
        avaliacao = "⚠ Cuidado — muitas partições"
    else:
        avaliacao = "✗ Evitar — cardinalidade altíssima"
    print(f"{col:<35} {n_unique:>18,}  {avaliacao}")

print()
print("Regra: cada partição deve ter entre 128 MB e 1 GB de dados.")
print("Poucas partições grandes → pouco benefício de pruning.")
print("Muitas partições pequenas → sobrecarga de metadados no NameNode/Metastore.")

size_parquet_mb = os.path.getsize('pedidos_snappy.parquet') / 1024**2
n_particoes     = df.groupby(['ano','mes']).ngroups
size_por_part   = size_parquet_mb / n_particoes

print(f"\nDataset atual:    {size_parquet_mb:.1f} MB em {n_particoes} partições ano+mes")
print(f"Tamanho médio/partição: {size_por_part:.2f} MB")
print("(Em produção, cada partição deveria ter 128 MB–1 GB;")
print(" nosso dataset é pequeno — o conceito é o mesmo)")

=== POR QUE PARTICIONAR POR ANO + MÊS? ===

COLUNA CANDIDATA                        VALORES ÚNICOS  AVALIAÇÃO
--------------------------------------------------------------------------------
ano                                                  3  ✓ Boa candidata
mes                                                 12  ✓ Boa candidata
ano+mes (combinado)                                 24  ✓ Boa candidata
dia                                                 31  ✓ Boa candidata
product_category_name                               73  ✓ Boa candidata
customer_state                                      27  ✓ Boa candidata
order_id (NÃO USAR)                             98,666  ✗ Evitar — cardinalidade altíssima
customer_id (NÃO USAR)                          98,666  ✗ Evitar — cardinalidade altíssima

Regra: cada partição deve ter entre 128 MB e 1 GB de dados.
Poucas partições grandes → pouco benefício de pruning.
Muitas partições pequenas → sobrecarga de metadados no NameNode/Metastore.

Dat

O trade-off central: order_id e customer_id têm cardinalidade altíssima — criariam dezenas de milhares de partições de kilobytes. Em um data lake real, listar todos esses arquivos no metastore custa mais do que varrer o arquivo monolítico. ano+mes com ~20 partições é o sweet spot para este dataset.

Parte 2 — Gravar os dados particionados
Célula 3 — Escrever estrutura Hive no disco

In [8]:
import shutil

if os.path.exists('pedidos_particionado'):
    shutil.rmtree('pedidos_particionado')
    print("Pasta anterior removida.")

print("Gravando estrutura particionada...\n")

t = time.time()

df.to_parquet(
    'pedidos_particionado/',
    partition_cols=['ano', 'mes'],
    index=False,
    compression='snappy'
)

t_escrita = time.time() - t
print(f"✓ Escrita concluída em {t_escrita:.2f}s")

n_arquivos  = 0
total_bytes = 0
for root, dirs, files in os.walk('pedidos_particionado/'):
    for f in files:
        if f.endswith('.parquet'):
            n_arquivos  += 1
            total_bytes += os.path.getsize(os.path.join(root, f))

print(f"Arquivos gerados:  {n_arquivos}")
print(f"Tamanho total:     {total_bytes/1024**2:.2f} MB")
print(f"Tamanho médio:     {total_bytes/1024**2/n_arquivos:.2f} MB por arquivo")

Gravando estrutura particionada...

✓ Escrita concluída em 0.15s
Arquivos gerados:  24
Tamanho total:     13.90 MB
Tamanho médio:     0.58 MB por arquivo


Uma única linha — partition_cols=['ano', 'mes'] — criou toda a estrutura. O pandas fez o split dos dados e gravou cada subconjunto no diretório correto. O tamanho total é próximo ao do arquivo monolítico: a estrutura Hive não aumenta o volume de dados, apenas muda como eles estão organizados no disco.

Célula 4 — Visualizar a estrutura de diretórios gerada

In [9]:
print("=== ESTRUTURA HIVE GERADA ===\n")
print("pedidos_particionado/")

total_arquivos = 0
total_tamanho  = 0

for root, dirs, files in sorted(os.walk('pedidos_particionado/')):
    nivel = root.replace('pedidos_particionado', '').count(os.sep)
    indent = '  ' * nivel
    pasta  = os.path.basename(root)

    parquet_files = [f for f in files if f.endswith('.parquet')]
    if parquet_files:
        tamanho_pasta = sum(
            os.path.getsize(os.path.join(root, f)) for f in parquet_files
        )
        total_tamanho  += tamanho_pasta
        total_arquivos += len(parquet_files)
        print(f"{indent}├── {pasta}/")
        for f in parquet_files:
            sz = os.path.getsize(os.path.join(root, f)) / 1024
            print(f"{indent}│   └── {f}  ({sz:.0f} KB)")
    elif nivel > 0 and not parquet_files:
        print(f"{indent}├── {pasta}/")

print(f"\nTotal: {total_arquivos} arquivos, {total_tamanho/1024**2:.2f} MB")
print()
print("Observe: a estrutura ano=XXXX/mes=X é a convenção Hive.")
print("O DuckDB, Spark e qualquer engine moderno reconhecem e usam")
print("esses nomes de diretório como filtros automáticos de partição.")

=== ESTRUTURA HIVE GERADA ===

pedidos_particionado/
  ├── /
  ├── ano=2016/
    ├── mes=10/
    │   └── ba424f7efe794ff2a618a68659acb4e0-0.parquet  (71 KB)
    ├── mes=12/
    │   └── ba424f7efe794ff2a618a68659acb4e0-0.parquet  (9 KB)
    ├── mes=9/
    │   └── ba424f7efe794ff2a618a68659acb4e0-0.parquet  (12 KB)
  ├── ano=2017/
    ├── mes=1/
    │   └── ba424f7efe794ff2a618a68659acb4e0-0.parquet  (143 KB)
    ├── mes=10/
    │   └── ba424f7efe794ff2a618a68659acb4e0-0.parquet  (666 KB)
    ├── mes=11/
    │   └── ba424f7efe794ff2a618a68659acb4e0-0.parquet  (1035 KB)
    ├── mes=12/
    │   └── ba424f7efe794ff2a618a68659acb4e0-0.parquet  (793 KB)
    ├── mes=2/
    │   └── ba424f7efe794ff2a618a68659acb4e0-0.parquet  (281 KB)
    ├── mes=3/
    │   └── ba424f7efe794ff2a618a68659acb4e0-0.parquet  (401 KB)
    ├── mes=4/
    │   └── ba424f7efe794ff2a618a68659acb4e0-0.parquet  (369 KB)
    ├── mes=5/
    │   └── ba424f7efe794ff2a618a68659acb4e0-0.parquet  (538 KB)
    ├── mes=6/
    │   └─

Esta é a estrutura Hive — inventada no Yahoo para o Hive sobre HDFS e hoje o padrão universal de data lakes. DuckDB, Spark, Athena, BigQuery e Redshift Spectrum a reconhecem sem configuração extra.

O nome do diretório ano=2018/mes=6/ já é o filtro. Quando a query tem WHERE ano=2018 AND mes=6, o engine monta o path diretamente e vai buscar apenas os arquivos dentro dessa pasta. As outras partições não chegam a ser abertas — nenhum byte é lido antes de saber exatamente o que buscar.

Parte 3 — Comparar leitura com e sem particionamento
Célula 5 — Query sem particionamento (baseline)

In [10]:
print("=== QUERY SEM PARTICIONAMENTO ===\n")
print("Arquivo: pedidos_snappy.parquet (único arquivo, todas as linhas)\n")

QUERY_SEM_PART = """
    SELECT
        product_category_name,
        COUNT(*)       AS num_pedidos,
        SUM(price)     AS receita_total,
        AVG(price)     AS ticket_medio
    FROM read_parquet('pedidos_snappy.parquet')
    WHERE ano = 2018 AND mes = 6
    GROUP BY product_category_name
    ORDER BY receita_total DESC
"""

tempos_sem = []
for i in range(3):
    t = time.time()
    resultado_sem = duckdb.sql(QUERY_SEM_PART).df()
    tempos_sem.append(time.time() - t)

t_sem = statistics.median(tempos_sem)

print(f"Tempo (mediana de 3): {t_sem:.4f}s")
print(f"Linhas retornadas:    {len(resultado_sem)}")
print()
print(resultado_sem.to_string(index=False))

=== QUERY SEM PARTICIONAMENTO ===

Arquivo: pedidos_snappy.parquet (único arquivo, todas as linhas)

Tempo (mediana de 3): 0.0037s
Linhas retornadas:    65

                  product_category_name  num_pedidos  receita_total  ticket_medio
                          health_beauty          885      107908.82        121.93
                          watches_gifts          481       86885.56        180.64
                         bed_bath_table          773       71565.48         92.58
                             housewares          584       56825.45         97.30
                         sports_leisure          426       45851.01        107.63
                                   auto          300       45407.49        151.36
                  computers_accessories          408       42326.45        103.74
                        furniture_decor          405       42190.22        104.17
                                   baby          225       30755.12        136.69
                       

A query filtra por ano=2018 AND mes=6, mas o arquivo contém dados de setembro de 2016 até outubro de 2018. O engine percorre o arquivo inteiro para encontrar as linhas do mês que nos interessa.

Célula 6 — Query com particionamento (com pruning)

In [11]:
print("=== QUERY COM PARTICIONAMENTO (hive_partitioning=true) ===\n")
print("Pasta: pedidos_particionado/ → engine vai direto em ano=2018/mes=6/\n")

QUERY_COM_PART = """
    SELECT
        product_category_name,
        COUNT(*)       AS num_pedidos,
        SUM(price)     AS receita_total,
        AVG(price)     AS ticket_medio
    FROM read_parquet(
        'pedidos_particionado/**/*.parquet',
        hive_partitioning = true
    )
    WHERE ano = 2018 AND mes = 6
    GROUP BY product_category_name
    ORDER BY receita_total DESC
"""

tempos_com = []
for i in range(3):
    t = time.time()
    resultado_com = duckdb.sql(QUERY_COM_PART).df()
    tempos_com.append(time.time() - t)

t_com = statistics.median(tempos_com)

print(f"Tempo (mediana de 3): {t_com:.4f}s")
print(f"Linhas retornadas:    {len(resultado_com)}")
print()
print(resultado_com.to_string(index=False))

resultados_iguais = resultado_sem.round(2).equals(resultado_com.round(2))
print(f"\n✓ Resultados idênticos: {resultados_iguais}")

=== QUERY COM PARTICIONAMENTO (hive_partitioning=true) ===

Pasta: pedidos_particionado/ → engine vai direto em ano=2018/mes=6/

Tempo (mediana de 3): 0.0102s
Linhas retornadas:    65

                  product_category_name  num_pedidos  receita_total  ticket_medio
                          health_beauty          885      107908.82        121.93
                          watches_gifts          481       86885.56        180.64
                         bed_bath_table          773       71565.48         92.58
                             housewares          584       56825.45         97.30
                         sports_leisure          426       45851.01        107.63
                                   auto          300       45407.49        151.36
                  computers_accessories          408       42326.45        103.74
                        furniture_decor          405       42190.22        104.17
                                   baby          225       30755.12        13

O parâmetro hive_partitioning=true é a chave. Sem ele, o DuckDB ignora os nomes de diretório — as colunas ano e mes nem existiriam nos arquivos Parquet (foram omitidas ao gravar). Com ele, o engine as reintroduz a partir do path e usa nos filtros.

Célula 7 — Tabela comparativa de tempo


In [12]:
print("\n" + "="*60)
print(f"{'COMPARAÇÃO DE PERFORMANCE':^60}")
print("="*60)

size_mono = os.path.getsize('pedidos_snappy.parquet') / 1024**2

part_path = None
for root, dirs, files in os.walk('pedidos_particionado/'):
    if 'ano=2018' in root and 'mes=6' in root:
        for f in files:
            if f.endswith('.parquet'):
                part_path = os.path.join(root, f)
                break

size_part = os.path.getsize(part_path) / 1024**2 if part_path else 0

print(f"\n{'':30} {'SEM PART.':>12}  {'COM PART.':>12}")
print("-"*60)
print(f"{'Arquivo(s) aberto(s)':<30} {'1 arquivo':>12}  {'1 de 20+':>12}")
print(f"{'Bytes lidos do disco':<30} {size_mono:>10.2f}MB  {size_part:>10.2f}MB")
print(f"{'Redução de I/O':<30} {'(baseline)':>12}  {(1-size_part/size_mono)*100:>10.1f}%")
print(f"{'Tempo mediano':<30} {t_sem:>11.4f}s  {t_com:>11.4f}s")

if t_com > 0:
    print(f"{'Speedup':<30} {'1.0x':>12}  {t_sem/t_com:>11.1f}x")

print("="*60)
print()
print("⚠  Com 112k linhas o ganho de tempo pode ser modesto.")
print("   O que importa é a REDUÇÃO DE I/O — os bytes lidos.")
print(f"   Sem partição: {size_mono:.2f} MB lidos.")
print(f"   Com partição: {size_part:.2f} MB lidos.")
print(f"   O engine ignorou {(1-size_part/size_mono)*100:.0f}% dos dados antes de abrir qualquer arquivo.")


                 COMPARAÇÃO DE PERFORMANCE                  

                                  SEM PART.     COM PART.
------------------------------------------------------------
Arquivo(s) aberto(s)              1 arquivo      1 de 20+
Bytes lidos do disco                10.93MB        0.87MB
Redução de I/O                   (baseline)        92.0%
Tempo mediano                       0.0037s       0.0102s
Speedup                                1.0x          0.4x

⚠  Com 112k linhas o ganho de tempo pode ser modesto.
   O que importa é a REDUÇÃO DE I/O — os bytes lidos.
   Sem partição: 10.93 MB lidos.
   Com partição: 0.87 MB lidos.
   O engine ignorou 92% dos dados antes de abrir qualquer arquivo.


O ganho de tempo aqui é de milissegundos — isso é esperado com 112k linhas. O que importa é a linha Bytes lidos do disco: a query com particionamento leu apenas uma fração dos dados. Com 1 TB de dados, essa fração equivale a centenas de gigabytes que não precisam ser transferidos.

Em serviços como AWS Athena você paga por TB de dados varridos. A economia pode chegar a 95–99% do custo de query dependendo do período filtrado e do tamanho do histórico.

Parte 4 — Tornar o pruning visível com EXPLAIN
O EXPLAIN mostra o plano de execução do DuckDB — o mesmo que um DBA usa para entender por que uma query está lenta.

Célula 8 — EXPLAIN: sem particionamento

In [13]:
print("=== PLANO DE EXECUÇÃO — SEM PARTICIONAMENTO ===\n")
print("O engine precisa abrir o arquivo e varrer todos os Row Groups")
print("para encontrar os registros de ano=2018 e mes=6\n")

explain_sem = duckdb.sql(f"EXPLAIN {QUERY_SEM_PART}").df()

for _, row in explain_sem.iterrows():
    print(row.iloc[1])

=== PLANO DE EXECUÇÃO — SEM PARTICIONAMENTO ===

O engine precisa abrir o arquivo e varrer todos os Row Groups
para encontrar os registros de ano=2018 e mes=6

┌───────────────────────────┐
│          ORDER_BY         │
│    ────────────────────   │
│  sum(read_parquet.price)  │
│            DESC           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│       HASH_GROUP_BY       │
│    ────────────────────   │
│         Groups: #0        │
│                           │
│        Aggregates:        │
│        count_star()       │
│          sum(#1)          │
│          avg(#2)          │
│                           │
│        ~20,419 rows       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│   product_category_name   │
│           price           │
│           price           │
│                           │
│        ~22,530 rows       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐


Observe onde aparece o filtro ano=2018 AND mes=6 no plano: ele está dentro do scan, aplicado sobre dados que o engine já foi buscar no arquivo. O arquivo foi aberto primeiro; os dados, lidos; o filtro, aplicado depois.

Célula 9 — EXPLAIN: com particionamento

In [14]:
print("=== PLANO DE EXECUÇÃO — COM PARTICIONAMENTO ===\n")
print("Observe: o engine filtra os arquivos pelo PATH antes de abrir qualquer um.")
print("'Filters: ano=2018, mes=6' aparece ANTES da leitura dos dados.\n")

explain_com = duckdb.sql(f"EXPLAIN {QUERY_COM_PART}").df()

for _, row in explain_com.iterrows():
    print(row.iloc[1])

=== PLANO DE EXECUÇÃO — COM PARTICIONAMENTO ===

Observe: o engine filtra os arquivos pelo PATH antes de abrir qualquer um.
'Filters: ano=2018, mes=6' aparece ANTES da leitura dos dados.

┌───────────────────────────┐
│          ORDER_BY         │
│    ────────────────────   │
│  sum(read_parquet.price)  │
│            DESC           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│       HASH_GROUP_BY       │
│    ────────────────────   │
│         Groups: #0        │
│                           │
│        Aggregates:        │
│        count_star()       │
│          sum(#1)          │
│          avg(#2)          │
│                           │
│         ~229 rows         │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│   product_category_name   │
│           price           │
│           price           │
│                           │
│         ~363 rows         │
└─────────────┬─────────────┘
┌─

Comparem os dois planos. No segundo, o filtro de partição aparece no momento de listar os arquivos — antes de qualquer leitura de dados. Isso é fundamentalmente diferente do predicate pushdown dentro de um arquivo Parquet (que opera em Row Groups). O partition pruning opera em nível de arquivo — ignora arquivos inteiros antes de abrir qualquer um. São dois mecanismos diferentes que se complementam.

Célula 10 — Listar explicitamente quais arquivos cada query abre

In [15]:
print("=== ARQUIVOS ABERTOS POR CADA ABORDAGEM ===\n")

print("SEM particionamento:")
print(f"  pedidos_snappy.parquet  ({size_mono:.2f} MB)  ← SEMPRE abre este")
print(f"  → contém dados de 2016-09 até 2018-10")
print(f"  → o engine filtra internamente por ano=2018/mes=6 após ler")
print()

print("COM particionamento — todos os arquivos disponíveis:")
todos_arquivos = []
for root, dirs, files in sorted(os.walk('pedidos_particionado/')):
    for f in sorted(files):
        if f.endswith('.parquet'):
            caminho = os.path.join(root, f)
            sz = os.path.getsize(caminho) / 1024
            todos_arquivos.append((caminho, sz))
            eh_relevante = 'ano=2018' in root and 'mes=6' in root
            marcador = '→ ABRE ✓' if eh_relevante else '→ ignora'
            print(f"  {caminho.replace('pedidos_particionado/',''):<45} ({sz:>5.0f} KB)  {marcador}")

total_kb = sum(sz for _, sz in todos_arquivos)
part_kb  = next((sz for path, sz in todos_arquivos
                 if 'ano=2018' in path and 'mes=6' in path), 0)

print()
print(f"Total disponível:  {total_kb/1024:.2f} MB em {len(todos_arquivos)} arquivos")
print(f"Efetivamente lido: {part_kb/1024:.2f} MB em 1 arquivo")
print(f"Ignorado:          {(total_kb-part_kb)/1024:.2f} MB ({(1-part_kb/total_kb)*100:.0f}% do total)")

=== ARQUIVOS ABERTOS POR CADA ABORDAGEM ===

SEM particionamento:
  pedidos_snappy.parquet  (10.93 MB)  ← SEMPRE abre este
  → contém dados de 2016-09 até 2018-10
  → o engine filtra internamente por ano=2018/mes=6 após ler

COM particionamento — todos os arquivos disponíveis:
  ano=2016/mes=10/ba424f7efe794ff2a618a68659acb4e0-0.parquet (   71 KB)  → ignora
  ano=2016/mes=12/ba424f7efe794ff2a618a68659acb4e0-0.parquet (    9 KB)  → ignora
  ano=2016/mes=9/ba424f7efe794ff2a618a68659acb4e0-0.parquet (   12 KB)  → ignora
  ano=2017/mes=1/ba424f7efe794ff2a618a68659acb4e0-0.parquet (  143 KB)  → ignora
  ano=2017/mes=10/ba424f7efe794ff2a618a68659acb4e0-0.parquet (  666 KB)  → ignora
  ano=2017/mes=11/ba424f7efe794ff2a618a68659acb4e0-0.parquet ( 1035 KB)  → ignora
  ano=2017/mes=12/ba424f7efe794ff2a618a68659acb4e0-0.parquet (  793 KB)  → ignora
  ano=2017/mes=2/ba424f7efe794ff2a618a68659acb4e0-0.parquet (  281 KB)  → ignora
  ano=2017/mes=3/ba424f7efe794ff2a618a68659acb4e0-0.parquet (  401 KB

De todos os arquivos disponíveis, a query de junho/2018 abre exatamente 1 — e ignora os outros sem sequer examiná-los. O mecanismo é direto: o engine monta o path pedidos_particionado/ano=2018/mes=6/ a partir dos valores no WHERE, lista os arquivos nessa pasta, e abre só eles.

Parte 5 — Experimentar diferentes estratégias de particionamento
Célula 11 — Particionamento por categoria

In [16]:
print("=== ALTERNATIVA: particionamento por product_category_name ===\n")

import shutil
if os.path.exists('pedidos_por_categoria'):
    shutil.rmtree('pedidos_por_categoria')

df.to_parquet(
    'pedidos_por_categoria/',
    partition_cols=['product_category_name'],
    index=False,
    compression='snappy'
)

print("Partições geradas:")
for root, dirs, files in sorted(os.walk('pedidos_por_categoria/')):
    for f in files:
        if f.endswith('.parquet'):
            caminho = os.path.join(root, f)
            sz = os.path.getsize(caminho) / 1024
            pasta = root.replace('pedidos_por_categoria/', '')
            print(f"  {pasta:<50} ({sz:>6.0f} KB)")

n_parts_cat = sum(
    1 for root, _, files in os.walk('pedidos_por_categoria/')
    for f in files if f.endswith('.parquet')
)
print(f"\nTotal: {n_parts_cat} arquivos (um por categoria)")

=== ALTERNATIVA: particionamento por product_category_name ===

Partições geradas:
  product_category_name=__HIVE_DEFAULT_PARTITION__   (   213 KB)
  product_category_name=agro_industry_and_commerce   (    45 KB)
  product_category_name=air_conditioning             (    56 KB)
  product_category_name=art                          (    45 KB)
  product_category_name=arts_and_craftmanship        (    22 KB)
  product_category_name=audio                        (    62 KB)
  product_category_name=auto                         (   531 KB)
  product_category_name=baby                         (   380 KB)
  product_category_name=bed_bath_table               (  1143 KB)
  product_category_name=books_general_interest       (    89 KB)
  product_category_name=books_imported               (    26 KB)
  product_category_name=books_technical              (    55 KB)
  product_category_name=cds_dvds_musicals            (    19 KB)
  product_category_name=christmas_supplies           (    37 KB)
  produ

Agora temos duas estruturas: pedidos_particionado/ (por ano+mes) e pedidos_por_categoria/ (~71 arquivos). Nenhuma é universalmente melhor — depende de qual coluna aparece mais nos filtros WHERE das queries do seu time.

Célula 12 — Quando cada particionamento faz sentido

In [17]:
print("=== COMPARAÇÃO: qual particionamento é melhor para qual query? ===\n")

q_data = """
    SELECT COUNT(*), SUM(price)
    FROM read_parquet('{path}', hive_partitioning=true)
    WHERE {filtro_data}
"""

q_cat = """
    SELECT COUNT(*), SUM(price)
    FROM read_parquet('{path}', hive_partitioning=true)
    WHERE {filtro_cat}
"""

print("─"*65)
print(f"{'QUERY':<35} {'PARTIÇÃO ANO+MES':>13}  {'PARTIÇÃO CAT':>12}")
print("─"*65)

t = time.time()
duckdb.sql(q_data.format(
    path='pedidos_particionado/**/*.parquet',
    filtro_data='ano = 2018 AND mes = 3'
))
t_data_anomes = time.time() - t

t = time.time()
duckdb.sql(q_data.format(
    path='pedidos_por_categoria/**/*.parquet',
    filtro_data='ano = 2018 AND mes = 3'
))
t_data_cat = time.time() - t

print(f"{'WHERE ano=2018 AND mes=3':<35} {t_data_anomes:>11.4f}s  {t_data_cat:>10.4f}s")
print(f"{'  → partition pruning?':<35} {'✓ SIM':>13}  {'✗ NÃO':>12}")

t = time.time()
duckdb.sql(q_cat.format(
    path='pedidos_por_categoria/**/*.parquet',
    filtro_cat="product_category_name = 'electronics'"
))
t_cat_cat = time.time() - t

t = time.time()
duckdb.sql(q_cat.format(
    path='pedidos_particionado/**/*.parquet',
    filtro_cat="product_category_name = 'electronics'"
))
t_cat_anomes = time.time() - t

print(f"{'WHERE category = electronics':<35} {t_cat_anomes:>11.4f}s  {t_cat_cat:>10.4f}s")
print(f"{'  → partition pruning?':<35} {'✗ NÃO':>13}  {'✓ SIM':>12}")
print("─"*65)
print()
print("Conclusão: a coluna de partição deve ser a coluna mais usada nos filtros.")
print("Não existe particionamento 'melhor' em abstrato — depende das queries.")

=== COMPARAÇÃO: qual particionamento é melhor para qual query? ===

─────────────────────────────────────────────────────────────────
QUERY                               PARTIÇÃO ANO+MES  PARTIÇÃO CAT
─────────────────────────────────────────────────────────────────
WHERE ano=2018 AND mes=3                 0.0376s      0.0086s
  → partition pruning?                      ✓ SIM         ✗ NÃO
WHERE category = electronics             0.0023s      0.0060s
  → partition pruning?                      ✗ NÃO         ✓ SIM
─────────────────────────────────────────────────────────────────

Conclusão: a coluna de partição deve ser a coluna mais usada nos filtros.
Não existe particionamento 'melhor' em abstrato — depende das queries.


A tabela resume o ponto mais importante do design de particionamento. Uma query com WHERE por data não ganha pruning numa estrutura particionada por categoria — o engine abre todos os 71 arquivos para encontrar os registros de março de 2018. O inverso também é verdadeiro.

Na prática, os times de dados consultam logs de queries antes de decidir o particionamento. O particionamento ideal é derivado desses dados, não de intuição.

Célula 13 — Granularidade adicional: ano + mês + dia

In [18]:
print("=== GRANULARIDADE DE PARTIÇÃO: até onde vai? ===\n")

import shutil
if os.path.exists('pedidos_diario'):
    shutil.rmtree('pedidos_diario')

df.to_parquet(
    'pedidos_diario/',
    partition_cols=['ano', 'mes', 'dia'],
    index=False,
    compression='snappy'
)

n_partes_dia = sum(
    1 for _, _, files in os.walk('pedidos_diario/')
    for f in files if f.endswith('.parquet')
)

tamanhos_dia = [
    os.path.getsize(os.path.join(root, f))
    for root, _, files in os.walk('pedidos_diario/')
    for f in files if f.endswith('.parquet')
]

print(f"Particionamento por category:    {n_parts_cat} partições")
print(f"Particionamento por ano+mes:     {df.groupby(['ano','mes']).ngroups} partições")
print(f"Particionamento por ano+mes+dia: {n_partes_dia} partições")
print()
print(f"Tamanho médio das partições diárias: {sum(tamanhos_dia)/len(tamanhos_dia)/1024:.1f} KB")
print()

print("ANÁLISE:")
print(f"  ano+mes:     {df.groupby(['ano','mes']).ngroups:>4} partições  → tamanho adequado para analytics")
print(f"  ano+mes+dia: {n_partes_dia:>4} partições  → muito granular para esse volume")
print()
print("Regra prática:")
print("  • Se queries filtram por mês → particionamento por ano+mes é suficiente")
print("  • Se queries filtram por dia → considere ano+mes+dia")
print("  • Cada partição deve ter entre 128 MB e 1 GB em produção")
print("  • Partições menores que 1 MB criam overhead de metadados")

q_dia = """
    SELECT COUNT(*), SUM(price)
    FROM read_parquet('{path}', hive_partitioning=true)
    WHERE ano = 2018 AND mes = 6 AND dia = 15
"""

t = time.time()
duckdb.sql(q_dia.format(path='pedidos_particionado/**/*.parquet'))
t_sem_dia = time.time() - t

t = time.time()
duckdb.sql(q_dia.format(path='pedidos_diario/**/*.parquet'))
t_com_dia = time.time() - t

print()
print(f"Query WHERE ano=2018 AND mes=6 AND dia=15:")
print(f"  Partição ano+mes:     {t_sem_dia:.4f}s  (abre partição de junho inteira)")
print(f"  Partição ano+mes+dia: {t_com_dia:.4f}s  (abre só o dia 15 de junho)")

=== GRANULARIDADE DE PARTIÇÃO: até onde vai? ===

Particionamento por category:    74 partições
Particionamento por ano+mes:     24 partições
Particionamento por ano+mes+dia: 616 partições

Tamanho médio das partições diárias: 45.6 KB

ANÁLISE:
  ano+mes:       24 partições  → tamanho adequado para analytics
  ano+mes+dia:  616 partições  → muito granular para esse volume

Regra prática:
  • Se queries filtram por mês → particionamento por ano+mes é suficiente
  • Se queries filtram por dia → considere ano+mes+dia
  • Cada partição deve ter entre 128 MB e 1 GB em produção
  • Partições menores que 1 MB criam overhead de metadados

Query WHERE ano=2018 AND mes=6 AND dia=15:
  Partição ano+mes:     0.0036s  (abre partição de junho inteira)
  Partição ano+mes+dia: 0.0431s  (abre só o dia 15 de junho)


Quanto mais granular a partição, mais preciso o pruning — mas mais arquivos o metastore precisa gerenciar. Com nosso dataset, as partições diárias têm alguns kilobytes: seria um anti-padrão em produção. O Databricks chama isso de small files problem — o Spark perde mais tempo gerenciando metadados do que processando dados.

Parte 6 — Simular escala: o que aconteceria com dados de produção

In [19]:
print("=== SE ESTE FOSSE UM DATASET DE PRODUÇÃO ===\n")

linhas_atual   = len(df)
size_mono_mb   = os.path.getsize('pedidos_snappy.parquet') / 1024**2
n_partic_atual = df.groupby(['ano','mes']).ngroups

fator_escala = 500_000_000 / linhas_atual

size_mono_gb   = size_mono_mb * fator_escala / 1024
size_part_gb   = size_mono_gb / (n_partic_atual * (5/2))

print(f"Dataset atual:        {linhas_atual:>12,} linhas  {size_mono_mb:.1f} MB")
print(f"Dataset projetado:    {500_000_000:>12,} linhas  {size_mono_gb:.0f} GB")
print()
print(f"{'OPERAÇÃO':<45} {'SEM PART.':>12}  {'COM PART.':>12}")
print("-"*72)
print(f"{'Tamanho lido (query 1 mês)':<45} {size_mono_gb:>10.0f}GB  {size_part_gb:>10.1f}GB")
print(f"{'Custo de query no Athena ($5/TB lido)':<45} "
      f"${size_mono_gb/1024*5:>9.2f}  ${size_part_gb/1024*5:>9.4f}")
print(f"{'Queries/mês (100 por dia × 30)':<45} {'3.000':>12}  {'3.000':>12}")
print(f"{'Custo mensal de queries (Athena)':<45} "
      f"${size_mono_gb/1024*5*3000:>9,.0f}  ${size_part_gb/1024*5*3000:>9,.2f}")
print()
print(f"Economia mensal com particionamento: "
      f"${(size_mono_gb-size_part_gb)/1024*5*3000:>,.0f} USD")
print()
print("Observação: Athena cobra $5 por TB de dados lidos.")
print("Particionamento correto pode reduzir o custo de queries em 90-99%.")

=== SE ESTE FOSSE UM DATASET DE PRODUÇÃO ===

Dataset atual:             112,650 linhas  10.9 MB
Dataset projetado:     500,000,000 linhas  47 GB

OPERAÇÃO                                         SEM PART.     COM PART.
------------------------------------------------------------------------
Tamanho lido (query 1 mês)                            47GB         0.8GB
Custo de query no Athena ($5/TB lido)         $     0.23  $   0.0039
Queries/mês (100 por dia × 30)                       3.000         3.000
Custo mensal de queries (Athena)              $      694  $    11.56

Economia mensal com particionamento: $682 USD

Observação: Athena cobra $5 por TB de dados lidos.
Particionamento correto pode reduzir o custo de queries em 90-99%.


O mesmo mecanismo que no nosso notebook reduz de 5 MB para 0.1 MB, numa empresa real pode reduzir de 500 GB para 10 GB por query. Multiplicado por 3.000 queries/mês, a economia chega a dezenas ou centenas de milhares de dólares por ano.

O modelo de precificação do Athena — e de outros sistemas como BigQuery — é literalmente "você paga pelo I/O". Particionamento correto é a forma mais direta de reduzir esse custo.